In [20]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Annotated
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.llms import Ollama

# Create the Ollama LLM
llm = Ollama(model="llama2")

In [21]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [22]:
def generate_joke(state: JokeState):
    prompt = f"generate joke on topic {state['topic']}"
    response= llm.invoke(prompt)

    return {"joke": response}

In [23]:
def generate_explanation(state: JokeState):
    prompt = f"generate explanation on joke {state['joke']}"
    response= llm.invoke(prompt)

    return {"explanation": response}

In [24]:
graph= StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [25]:
thread_id='1'
config ={
        "configurable":{
            "thread_id": thread_id
        }
    }

In [26]:
workflow.invoke({'topic':'pizza'},config=config)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to therapy?\nIt was feeling a little crusty!',
 'explanation': '\n Ah, I see! The joke is a play on words, with "crusty" having both its literal meaning (the outside layer of a pizza) and a metaphorical meaning (feeling mentally unstable or stressed). The pizza, in this case, is seeking therapy because it\'s feeling a little crusty, i.e., mentally unstable or stressed. It\'s a funny and creative way to use language to create a humorous scenario!'}

In [27]:
#Timetravel